# Computation of intermadiate reward in trajectory level aggregation 

Guida che mostra come vengono aggunti  i reward intermedi per il training con GRPO

In [1]:
import torch
from typing import List, Tuple, Any

In [ ]:
# immaginiamo che abbiamo tre triplette ognuna contentente: {prompt_ids, response_ids, reward}
triplet_1 = {
    "prompt_ids": [4,3,1,4],
    "response_ids": [3,2,4],
    "reward": 0.0
}
triplet_2 = {
    "prompt_ids": [4,3,1,4,3,2,4,4,3,1],
    "response_ids": [9,1,3],
    "reward": 1.0
}
triplet_3 = {
    "prompt_ids": [4,3,1,4,3,2,4,4,3,1,9,1,3,7,8],
    "response_ids": [1,2,3],
    "reward": 0
}

triplets = [triplet_1,triplet_2, triplet_3]


In [48]:
# bisogna unire i triplet contigui in un unico training sample
def ids_startswith(
    full_ids: List[int], prefix_ids: List[int]
) -> bool:
    
    if full_ids[: len(prefix_ids)] == prefix_ids:
        
        return True
    else:
         return False 


merged_trace_idx: List[List[int]] = []

current_merged_trace_idx: List[int] = []
current_context: List[int] = []

for turn_index,trace in enumerate(triplets):
    is_prefix = ids_startswith(
                        trace["prompt_ids"] + trace["response_ids"],
                        current_context
                    )
    
    if is_prefix:
        current_context = trace["prompt_ids"] + trace["response_ids"]
        current_merged_trace_idx.append(turn_index)

    
if current_merged_trace_idx not in merged_trace_idx:
    merged_trace_idx.append(current_merged_trace_idx)


# rappresenta l'ordine dei indici nella rollout da mergare assieme
merged_trace_idx

[[0, 1, 2]]

In [ ]:
# adesso si fa il merge tenendo conto la struttura: il prompt rimane solo la prima triplet, il resto è response
max_prompt_length = 8
for current_merged_trace_idx in merged_trace_idx:
    prompt_ids = triplets[current_merged_trace_idx[0]]["prompt_ids"]
    # if the merged_trace_idx doesn't start with the beginning of the prompt_ids, we need to adjust it
    if current_merged_trace_idx[0] > 0 and len(prompt_ids) > max_prompt_length:
        response_ids = prompt_ids[max_prompt_length:]
        prompt_ids = prompt_ids[:max_prompt_length]
        response_mask = [1] * len(response_ids)
        # add also to the reward mask
        reward_mask = [0] * len(response_ids)
    else:
        response_ids = []
        response_mask = []
        reward_mask = []

    prompt_length = len(prompt_ids)
    response_ids += triplets[current_merged_trace_idx[0]]["response_ids"]
    response_mask += [1] * len(response_ids)
    # add to the reward mask with specific intermediate reward in the last position
    reward_mask += [0] * len(response_ids)
    reward_mask[-1] = triplets[current_merged_trace_idx[0]]["reward"]
    
    print(f"prompt_ids  prima del for: {prompt_ids}")
    print(f"response_ids  prima del for: {response_ids}")
    print(f"response_mask  prima del for: {response_mask}")
    print(f"reward_mask  prima del for: {reward_mask}")
    for turn_index in current_merged_trace_idx[1:]:
        trace = triplets[turn_index]
        new_prompt_length = len(trace["prompt_ids"]) - len(response_ids) - prompt_length

        print(f"new_prompt_length al turno {turn_index}: {new_prompt_length}")
        response_ids += trace["prompt_ids"][-new_prompt_length:]
        response_ids += trace["response_ids"]
        response_mask += [0] * new_prompt_length
        response_mask += [1] * len(trace["response_ids"])
        # add to the reward mask
        reward_mask += [0] * (len(trace["response_ids"]) + new_prompt_length)
        reward_mask[-1] = trace["reward"]

        print(f"response_ids al turno {turn_index}: {response_ids}")
        print(f"response_mask al turno {turn_index}: {response_mask}")
        print(f"reward_mask al turno {turn_index}: {reward_mask}")

prompt_ids  prima del for: [4, 3, 1, 4]
response_ids  prima del for: [3, 2, 4]
response_mask  prima del for: [1, 1, 1]
reward_mask  prima del for: [0, 0, 0.0]
new_prompt_length al turno 1: 3
response_ids al turno 1: [3, 2, 4, 4, 3, 1, 9, 1, 3]
response_mask al turno 1: [1, 1, 1, 0, 0, 0, 1, 1, 1]
reward_mask al turno 1: [0, 0, 0.0, 0, 0, 0, 0, 0, 1.0]
new_prompt_length al turno 2: 2
response_ids al turno 2: [3, 2, 4, 4, 3, 1, 9, 1, 3, 7, 8, 1, 2, 3]
response_mask al turno 2: [1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1]
reward_mask al turno 2: [0, 0, 0.0, 0, 0, 0, 0, 0, 1.0, 0, 0, 0, 0, 0]


In [50]:
# allineiamo il prompt a destra ( il testo importante è alla fine)
# allineiamo la response,la response_mask e la reward_mask a sinistra (il testo importante e all'inizio)
def get_left_padded_ids_and_attention_mask(
    ids: List[int], max_length: int, pad_token_id: int
) -> Tuple[List[int], List[int]]:
    """
    Left-pad (or truncate) a sequence of token IDs to a fixed length,
    and build the corresponding attention mask.

    Args:
        ids:             the original list of token IDs.
        max_length:      desired total length after padding/truncation.
        pad_token_id:    ID to use for padding.

    Returns:
        padded_ids (any):      list of length == max_length.
        attention_mask (any):  list of same length: 1 for non-pad tokens, 0 for pads.
    """
    seq_len = len(ids)

    if seq_len >= max_length:
        # too long → truncate from the left, keep the last max_length tokens
        trimmed = ids[-max_length:]
        attention_mask = [1] * max_length
        return trimmed, attention_mask

    # too short → pad on the left
    pad_len = max_length - seq_len
    padded_ids = [pad_token_id] * pad_len + ids
    attention_mask = [0] * pad_len + [1] * seq_len
    return padded_ids, attention_mask

def get_right_padded_ids_and_attention_mask(
    ids: List[Any], max_length: int, pad_token_id: int
) -> Tuple[List[Any], List[Any]]:
    """
    Right-pad (or truncate) a sequence of token IDs to a fixed length,
    and build the corresponding attention mask.

    Args:
        ids:            the original list of token IDs.
        max_length:     desired total length after padding/truncation.
        pad_token_id:   ID to use for padding.

    Returns:
        padded_ids (any):     list of length == max_length.
        attention_mask (any): list of same length: 1 for non-pad tokens, 0 for pads.
    """
    seq_len = len(ids)

    if seq_len >= max_length:
        # too long → truncate to the first max_length tokens
        trimmed = ids[:max_length]
        attention_mask = [1] * max_length
        return trimmed, attention_mask

    # too short → pad on the right
    pad_len = max_length - seq_len
    padded_ids = ids + [pad_token_id] * pad_len
    attention_mask = [1] * seq_len + [0] * pad_len
    return padded_ids, attention_mask

# bisogna paddare fino alla max_response_length
max_response_length = 20

one_input_ids, one_input_attention_mask = get_left_padded_ids_and_attention_mask(
    prompt_ids, max_prompt_length, 0
)
one_response_ids, one_response_attention_mask = get_right_padded_ids_and_attention_mask(
    response_ids, max_response_length, 0
)
one_reward_mask, _ = get_right_padded_ids_and_attention_mask(
    reward_mask, max_response_length, 0, 
)
one_response_mask, _ = get_right_padded_ids_and_attention_mask(
    response_mask, max_response_length, 0
)

print(f"""
one_input_ids: {one_input_ids}
one_input_attention_mask: {one_input_attention_mask}

one_response_ids: {one_response_ids}
one_response_attention_mask: {one_response_attention_mask}

one_reward_mask: {one_reward_mask}

one_response_mask: {one_response_mask}
""")


one_input_ids: [0, 0, 0, 0, 4, 3, 1, 4]
one_input_attention_mask: [0, 0, 0, 0, 1, 1, 1, 1]

one_response_ids: [3, 2, 4, 4, 3, 1, 9, 1, 3, 7, 8, 1, 2, 3, 0, 0, 0, 0, 0, 0]
one_response_attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]

one_reward_mask: [0, 0, 0.0, 0, 0, 0, 0, 0, 1.0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

one_response_mask: [1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0]



In [51]:
#  aggeghiamo per batch con le altre rollout
input_ids_list: List[List[int]] = []
input_attention_mask_list: List[List[int]] = []
response_ids_list: List[List[int]] = []
response_attention_mask_list: List[List[int]] = []
reward_list: List[float] = []
reward_mask_list: List[List[float]] = []
response_mask_list: List[List[int]] = []

input_ids_list.append(one_input_ids)
input_attention_mask_list.append(one_input_attention_mask)
response_ids_list.append(one_response_ids)
response_attention_mask_list.append(one_response_attention_mask)
response_mask_list.append(one_response_mask)
reward_mask_list.append(one_reward_mask)

In [52]:
# trasferiamo tutti i samples sulla gpu e aggreghiamo i prompt e le response per riga
device = 'cpu'
batch_input_ids = torch.LongTensor(input_ids_list).to(device)
input_attention_mask = torch.LongTensor(input_attention_mask_list).to(device)
batch_response_ids = torch.LongTensor(response_ids_list).to(device)
response_attention_mask = torch.LongTensor(response_attention_mask_list).to(device)
response_mask = torch.LongTensor(response_mask_list).to(device)
reward_batch = torch.tensor(reward_mask_list, dtype=torch.float16).to(device)

# Concatenate prompts and responses to form the full sequence
batch_seq = torch.cat([batch_input_ids, batch_response_ids], dim=-1)
attention_mask = torch.cat([input_attention_mask, response_attention_mask], dim=-1)

print(batch_seq)
print(attention_mask)

tensor([[0, 0, 0, 0, 4, 3, 1, 4, 3, 2, 4, 4, 3, 1, 9, 1, 3, 7, 8, 1, 2, 3, 0, 0,
         0, 0, 0, 0]])
tensor([[0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0,
         0, 0, 0, 0]])


In [53]:
# calcoliamo i positions_ids -> posizione progressia dei token non paddati
position_ids = torch.clamp(torch.cumsum(attention_mask, dim=-1) - 1, min=0)

position_ids

tensor([[ 0,  0,  0,  0,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13,
         14, 15, 16, 17, 17, 17, 17, 17, 17, 17]])

In [55]:
# adesso si crea il token level score da passare a verl
token_level_scores = torch.zeros_like(attention_mask[:,-max_response_length:], dtype=torch.bfloat16)

# summ with reward_mask * attention_mask
token_level_scores += reward_batch * attention_mask[:,-max_response_length:]

token_level_scores

tensor([[0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0.]], dtype=torch.bfloat16)